# Reto práctico: analizar el rendimiento diario de una cadena de tiendas

## Contexto del problema

Una cadena de dos tiendas quiere revisar su rendimiento durante una semana. Tiene ventas de dos canales por tienda y día. El equipo quiere un informe que permita ver la evolución acumulada, comparar cada día con su entorno cercano y medir el comportamiento reciente.

Tu trabajo es construir ese informe con SQL. Todas las consultas deben poder ejecutarse desde cero con los datos incluidos en este notebook.

## Datos disponibles

La tabla `ventas` contiene registros sintéticos. Cada fila representa las ventas de un canal para una tienda y un día.

| Columna | Significado |
|---|---|
| `event_id` | Identificador del registro |
| `store` | Tienda |
| `sale_date` | Día de venta |
| `channel` | Canal de venta |
| `sales` | Unidades vendidas |

Hay dos registros por tienda y día, uno por canal. Por tanto, una misma fecha aparece varias veces dentro de una tienda. Esa característica forma parte del problema y debe tenerse en cuenta al analizar el resultado.

El reto usa una base de datos embebida para que no necesites una conexión externa ni ningún recurso fuera del notebook.

In [1]:
import sqlite3

# Recurso técnico del notebook: base de datos embebida y datos sintéticos.
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row

conn.execute("""
CREATE TABLE ventas (
    event_id INTEGER PRIMARY KEY,
    store TEXT NOT NULL,
    sale_date TEXT NOT NULL,
    channel TEXT NOT NULL,
    sales INTEGER NOT NULL
)
""")

ventas = [
    (1,  'Centro', '2025-01-01', 'web',    120),
    (2,  'Centro', '2025-01-01', 'tienda',  80),
    (3,  'Centro', '2025-01-02', 'web',    140),
    (4,  'Centro', '2025-01-02', 'tienda',  95),
    (5,  'Centro', '2025-01-03', 'web',    125),
    (6,  'Centro', '2025-01-03', 'tienda', 110),
    (7,  'Centro', '2025-01-04', 'web',    160),
    (8,  'Centro', '2025-01-04', 'tienda', 105),
    (9,  'Centro', '2025-01-05', 'web',    150),
    (10, 'Centro', '2025-01-05', 'tienda', 120),
    (11, 'Centro', '2025-01-06', 'web',    130),
    (12, 'Centro', '2025-01-06', 'tienda', 125),
    (13, 'Centro', '2025-01-07', 'web',    145),
    (14, 'Centro', '2025-01-07', 'tienda', 135),
    (15, 'Norte',  '2025-01-01', 'web',     90),
    (16, 'Norte',  '2025-01-01', 'tienda',  70),
    (17, 'Norte',  '2025-01-02', 'web',    100),
    (18, 'Norte',  '2025-01-02', 'tienda',  75),
    (19, 'Norte',  '2025-01-03', 'web',    105),
    (20, 'Norte',  '2025-01-03', 'tienda',  80),
    (21, 'Norte',  '2025-01-04', 'web',     95),
    (22, 'Norte', '2025-01-04', 'tienda',  90),
    (23, 'Norte', '2025-01-05', 'web',    120),
    (24, 'Norte', '2025-01-05', 'tienda',  85),
    (25, 'Norte', '2025-01-06', 'web',    115),
    (26, 'Norte', '2025-01-06', 'tienda',  95),
    (27, 'Norte', '2025-01-07', 'web',    130),
    (28, 'Norte', '2025-01-07', 'tienda', 100),
]

conn.executemany(
    "INSERT INTO ventas (event_id, store, sale_date, channel, sales) VALUES (?, ?, ?, ?, ?)",
    ventas,
)
conn.commit()

def show(query, params=()):
    rows = conn.execute(query, params).fetchall()
    if not rows:
        print('(sin filas)')
        return
    headers = rows[0].keys()
    data = [list(row) for row in rows]
    widths = [len(str(h)) for h in headers]
    for row in data:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(str(value)))
    line = '-+-'.join('-' * w for w in widths)
    def fmt(value):
        if isinstance(value, float):
            return f'{value:.2f}'
        return str(value)
    formatted_rows = [[fmt(value) for value in row] for row in data]
    for row in formatted_rows:
        for i, value in enumerate(row):
            widths[i] = max(widths[i], len(value))
    line = '-+-'.join('-' * w for w in widths)
    print(' | '.join(str(h).ljust(widths[i]) for i, h in enumerate(headers)))
    print(line)
    for row in formatted_rows:
        print(' | '.join(value.ljust(widths[i]) for i, value in enumerate(row)))

show("""
SELECT event_id, store, sale_date, channel, sales
FROM ventas
ORDER BY store, sale_date, event_id;
""")

event_id | store  | sale_date  | channel | sales
---------+--------+------------+---------+------
1        | Centro | 2025-01-01 | web     | 120  
2        | Centro | 2025-01-01 | tienda  | 80   
3        | Centro | 2025-01-02 | web     | 140  
4        | Centro | 2025-01-02 | tienda  | 95   
5        | Centro | 2025-01-03 | web     | 125  
6        | Centro | 2025-01-03 | tienda  | 110  
7        | Centro | 2025-01-04 | web     | 160  
8        | Centro | 2025-01-04 | tienda  | 105  
9        | Centro | 2025-01-05 | web     | 150  
10       | Centro | 2025-01-05 | tienda  | 120  
11       | Centro | 2025-01-06 | web     | 130  
12       | Centro | 2025-01-06 | tienda  | 125  
13       | Centro | 2025-01-07 | web     | 145  
14       | Centro | 2025-01-07 | tienda  | 135  
15       | Norte  | 2025-01-01 | web     | 90   
16       | Norte  | 2025-01-01 | tienda  | 70   
17       | Norte  | 2025-01-02 | web     | 100  
18       | Norte  | 2025-01-02 | tienda  | 75   
19       | Norte  | 

## Objetivo

Entrega un informe que permita a cada tienda responder estas preguntas con sus datos diarios: cómo evoluciona el acumulado, cuál ha sido su mejor y peor día hasta cada fecha, cómo se comporta la actividad reciente y cómo cambia la lectura cuando una fecha tiene varias filas.

El resultado final debe ser reproducible con los datos incluidos en este notebook.

## Requisitos del reto

Construye una consulta o un conjunto de consultas relacionadas que formen un único informe. No se pide una técnica concreta: decide tú cómo organizar la solución.

1. Para cada tienda y día, muestra las ventas diarias, el total acumulado hasta esa fecha, el máximo acumulado hasta esa fecha, el mínimo acumulado hasta esa fecha y cuántos días se han observado hasta ese momento. Añade también el máximo de ventas de toda la serie de cada tienda, repetido en cada fila.

2. Añade dos indicadores recientes para cada tienda y día: el promedio de ventas de los últimos tres días disponibles, contando el día actual, y la suma de esas mismas ventas.

3. Añade dos indicadores de máximo local: uno que use el día anterior y el actual, y otro que use el actual y el siguiente. Cuando no exista una fila anterior o siguiente, el resultado debe ajustarse a las filas que realmente existan.

4. Añade un valor de referencia basado solo en las cinco filas anteriores al día actual, sin incluir el día actual. Para los primeros días puede no existir un valor suficiente y el resultado puede quedar vacío.

5. En una salida aparte, trabaja con la tabla original `ventas` y compara un total acumulado calculado por filas con otro acumulado que trate las fechas repetidas como una sola entidad. Debes conservar la tienda, la fecha, el canal y las dos lecturas acumuladas para poder explicar la diferencia.

6. La lectura final debe resetear los cálculos por tienda. Ordena los resultados cronológicamente dentro de cada tienda.

### Criterios de aceptación

- Los cálculos respetan el orden temporal.
- Los acumulados se reinician al cambiar de tienda.
- Los indicadores de tres días usan como máximo el día actual y los dos días anteriores.
- El indicador que mira hacia delante usa como máximo el día actual y el siguiente.
- El valor de referencia no incorpora el día actual.
- La comparación final permite ver el efecto de las fechas duplicadas.
- Todo se puede ejecutar de nuevo sin depender de un estado anterior de la base de datos.

In [2]:
# ESPACIO DE RESOLUCIÓN
# Escribe y prueba aquí tu solución.
# Puedes crear tantas consultas y celdas adicionales como necesites.

# Solución - consultar después de intentar

La solución se divide en dos partes del mismo reto: el informe diario por tienda y la comparación sobre las filas originales. Se mantienen todos los indicadores dentro de la misma base de datos y con los mismos datos sintéticos.

In [3]:
# Parte 1: informe diario por tienda.
# Primero se consolida la venta de los dos canales en una fila por tienda y día.
q_daily = """
WITH daily_sales AS (
    SELECT
        store,
        sale_date,
        SUM(sales) AS daily_sales
    FROM ventas
    GROUP BY store, sale_date
)
SELECT
    store,
    sale_date,
    daily_sales,
    SUM(daily_sales) OVER (
        PARTITION BY store
        ORDER BY sale_date ASC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS cumulative_sales,
    MAX(daily_sales) OVER (
        PARTITION BY store
        ORDER BY sale_date ASC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS max_so_far,
    MIN(daily_sales) OVER (
        PARTITION BY store
        ORDER BY sale_date ASC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS min_so_far,
    COUNT(*) OVER (
        PARTITION BY store
        ORDER BY sale_date ASC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS days_seen,
    MAX(daily_sales) OVER (
        PARTITION BY store
        ORDER BY sale_date ASC
        RANGE BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS full_period_max,
    AVG(daily_sales) OVER (
        PARTITION BY store
        ORDER BY sale_date ASC
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS avg_last_3_days,
    SUM(daily_sales) OVER (
        PARTITION BY store
        ORDER BY sale_date ASC
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS total_last_3_days,
    MAX(daily_sales) OVER (
        PARTITION BY store
        ORDER BY sale_date ASC
        ROWS BETWEEN 1 PRECEDING AND CURRENT ROW
    ) AS max_previous_current,
    MAX(daily_sales) OVER (
        PARTITION BY store
        ORDER BY sale_date ASC
        ROWS BETWEEN CURRENT ROW AND 1 FOLLOWING
    ) AS max_current_next,
    MAX(daily_sales) OVER (
        PARTITION BY store
        ORDER BY sale_date ASC
        ROWS BETWEEN 5 PRECEDING AND 1 PRECEDING
    ) AS max_previous_5_only
FROM daily_sales
ORDER BY store, sale_date;
"""

# Parte 2: comparación sobre las filas originales, donde cada fecha aparece dos veces.
q_rows_vs_range = """
SELECT
    event_id,
    store,
    sale_date,
    channel,
    sales,
    SUM(sales) OVER (
        PARTITION BY store
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS rows_running_total,
    SUM(sales) OVER (
        PARTITION BY store
        ORDER BY sale_date
        RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS range_running_total
FROM ventas
ORDER BY store, sale_date, event_id;
"""

# Ejecución real de las consultas de la solución.
daily_result = conn.execute(q_daily).fetchall()
range_result = conn.execute(q_rows_vs_range).fetchall()

## Resultados obtenidos

Los siguientes resultados son los producidos al ejecutar las consultas anteriores con los datos sintéticos del notebook.

In [4]:
print('INFORME DIARIO POR TIENDA')
print()
show(q_daily)

print()
print('COMPARACIÓN ROWS VS RANGE')
print()
show(q_rows_vs_range)

INFORME DIARIO POR TIENDA

store  | sale_date  | daily_sales | cumulative_sales | max_so_far | min_so_far | days_seen | full_period_max | avg_last_3_days    | total_last_3_days | max_previous_current | max_current_next | max_previous_5_only
-------+------------+-------------+------------------+------------+------------+-----------+-----------------+--------------------+-------------------+----------------------+------------------+--------------------
Centro | 2025-01-01 | 200         | 200              | 200        | 200        | 1         | 280             | 200.00             | 200               | 200                  | 235              | None               
Centro | 2025-01-02 | 235         | 435              | 235        | 200        | 2         | 280             | 217.50             | 435               | 235                  | 235              | 200                
Centro | 2025-01-03 | 235         | 670              | 235        | 200        | 3         | 280             | 223.33

## Interpretación y autoevaluación

En el informe diario, los acumulados avanzan dentro de cada tienda y se reinician al comenzar la otra. La media y la suma de tres días solo incorporan las filas que existen en ese punto de la serie, por lo que los primeros días tienen ventanas más cortas. Los indicadores con anterior y siguiente cambian el conjunto de filas que se toma como referencia, mientras que el valor de los cinco anteriores excluye el día actual.

En la comparación final, `ROWS` procesa cada registro de la secuencia. Como una fecha tiene dos registros, su acumulado puede avanzar entre ambos. `RANGE` trata los registros con la misma fecha del `ORDER BY` como una misma entidad y muestra el mismo acumulado para ellos. Esto es exactamente el comportamiento que la documentación destaca para distinguir `ROWS` de `RANGE`.

### Autoevaluación

Comprueba tu trabajo con estas preguntas:

- ¿Separaste los cálculos por tienda y mantuviste el orden temporal?
- ¿Tus ventanas de tres días incluyen como máximo los dos días anteriores y el actual?
- ¿Tu ventana hacia delante puede usar el siguiente día sin inventar una fila cuando no existe?
- ¿Tu referencia de cinco anteriores excluye el día actual?
- ¿Puedes explicar por qué `ROWS` y `RANGE` producen lecturas distintas cuando una fecha aparece varias veces?
- ¿Has usado de forma intencionada marcos completos, acumulados, ventanas hacia atrás y ventanas hacia delante, en lugar de obtener solo el resultado final?